# Inference Suboptimality on a Kaggle GPU

Before running, in the right-hand panel under **Session options**:
- **Accelerator:** `GPU T4` (the code uses a single GPU; the P100 is not supported by recent PyTorch builds)
- **Internet:** On (needed for `git clone` and the CIFAR download; requires a phone-verified account)

The experiment is defined by `config.yaml` in the repository: edit it locally, commit and push,
and this notebook picks it up.

**Running without keeping the browser open:** use **Save Version → Save & Run All (Commit)**.
That runs the whole notebook in a fresh background session (up to 12 h), so you can close the tab.
It clones the branch as pushed to GitHub, so push your changes first. Follow progress under
**View Active Events**; when it finishes, `/kaggle/working` becomes the version's **Output**.
An interactive session stops when you close the tab or stay idle too long.

In [ ]:
# --- Configuration -----------------------------------------------------------
REPO = "https://github.com/ugwnate-hash/inference-suboptimality.git"
BRANCH = "my-work"
CONFIG = "config.yaml"   # experiment settings, relative to the repository root

## 1. Get the code

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.isdir("inference-suboptimality"):
    !git clone -b {BRANCH} {REPO}
else:
    !git -C inference-suboptimality pull
os.chdir("/kaggle/working/inference-suboptimality")
!git log --oneline -1

## 2. Check the GPU and install dependencies

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0))
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("WARNING: this PyTorch build does not support this GPU; switch the accelerator to GPU T4")
else:
    print("WARNING: no GPU; enable it under Session options -> Accelerator")

!pip install -q -r requirements.txt

In [ ]:
# Load the experiment config; the scripts read the same file via --config
import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
DATASET = cfg["dataset"]
print(open(CONFIG).read())

## 3. Train

All settings come from `config.yaml`; a flag added to a command overrides the file.
Checkpoints go to `checkpoints/<dataset>/<warmup>/<model>/<epoch>_model.pth` and the log to `logs/`.
`best_model.pth` in the same folder is the model with the best validation ELBO.
To resume an interrupted run, add `--load-path <ckpt> --load-epoch <epoch>` to the command;
`epochs` is the total, and warmup and the learning-rate schedule carry on from `<epoch>`.
`TQDM_DISABLE=1` turns off the per-iteration progress bars, which would otherwise flood the log.

In [ ]:
!mkdir -p logs
!TQDM_DISABLE=1 python -u run.py --train --config {CONFIG} 2>&1 | tee -a logs/train_{DATASET}.log

In [ ]:
# Pick the best model of the most recent run (or set CKPT by hand)
import glob
checkpoints = glob.glob(f"checkpoints/{DATASET}/**/*_model.pth", recursive=True)
latest = max(checkpoints, key=os.path.getmtime)
best = os.path.join(os.path.dirname(latest), "best_model.pth")
CKPT = best if os.path.exists(best) else latest
print("Evaluating", CKPT)

## 4. Evaluate

Sample counts (`n_iwae`, `n_ais_dist`, `n_ais_iwae`) are set in the `eval` section of `config.yaml`.
They default to moderate values so each cell finishes in reasonable time. The paper's settings are
`n_iwae: 5000`, `n_ais_dist: 10000` and `n_ais_iwae: 100`; increase gradually.

In [ ]:
# ELBO (100 MC samples) and IWAE bound with the amortized encoder
!TQDM_DISABLE=1 python -u run.py --eval-iwae --config {CONFIG} --eval-path {CKPT} 2>&1 | tee logs/eval_iwae_{DATASET}.log

In [ ]:
# AIS estimate of log p(x) (close to the true marginal likelihood)
!TQDM_DISABLE=1 python -u run.py --eval-ais --config {CONFIG} --eval-path {CKPT} 2>&1 | tee logs/eval_ais_{DATASET}.log

In [ ]:
# Locally optimized Gaussian per datapoint (removes the amortization gap)
!TQDM_DISABLE=1 python -u local_ffg.py --config {CONFIG} --eval-path {CKPT} 2>&1 | tee logs/local_ffg_{DATASET}.log

In [ ]:
# Locally optimized flow per datapoint (slow: one datapoint at a time)
# !TQDM_DISABLE=1 python -u local_flow.py --config {CONFIG} --eval-path {CKPT} 2>&1 | tee logs/local_flow_{DATASET}.log

In [ ]:
# Bidirectional Monte Carlo on simulated data: checks how tight AIS is
!TQDM_DISABLE=1 python -u bdmc.py --config {CONFIG} --eval-path {CKPT} 2>&1 | tee logs/bdmc_{DATASET}.log

## 5. Plot the posteriors (2-D latent models only)

Shows the true posterior p(z|x) (the target) against the encoder's q(z|x) and a Gaussian
optimized per image, for the untrained model, the first checkpoint and the best one.
The true posterior can only be computed on a grid in 2-D, so this section needs
`z_size: 2` in `config.yaml`.

In [ ]:
from IPython.display import Image, display
if cfg.get("model", {}).get("z_size") == 2:
    FIRST = os.path.join(os.path.dirname(CKPT), "1_model.pth")
    !python -u plot_posteriors.py --config {CONFIG} --checkpoints init {FIRST} {CKPT} --out plots/posteriors_{DATASET}.png 2>&1 | tee logs/plot_posteriors_{DATASET}.log
    display(Image(f"plots/posteriors_{DATASET}.png"))
else:
    print("Skipped: needs z_size: 2 in config.yaml")

## 6. Package results

Everything under `/kaggle/working` is kept as this version's output. The archive below is a single
file to download, or to attach to another notebook via **Output → New Dataset**.

In [ ]:
!cp {CONFIG} logs/ && mkdir -p plots
!tar -czf /kaggle/working/results_{DATASET}.tar.gz checkpoints logs plots
!ls -lh /kaggle/working/results_{DATASET}.tar.gz